In [8]:
import os

print(os.getcwd())

import pandas as pd

# =====================================================
# LOAD DATASETS
# =====================================================

payroll = pd.read_csv("NBA_team_payroll_2026.csv")
salarycap = pd.read_csv("NBA_team_salarycap_2026.csv")
stats = pd.read_csv("NBA_team_stats.csv")
advanced = pd.read_csv("NBA_team_stats2.csv")

# =====================================================
# CLEAN TEAM NAMES
# =====================================================

# Remove asterisks from Basketball Reference playoff markers
stats["Team"] = stats["Team"].str.replace("*", "", regex=False)
advanced["Team"] = advanced["Team"].str.replace("*", "", regex=False)

# =====================================================
# CREATE TEAM NAME MAPPING
# =====================================================

team_map = {
    "Atlanta Hawks": "ATL",
    "Boston Celtics": "BOS",
    "Brooklyn Nets": "BKN",
    "Charlotte Hornets": "CHA",
    "Chicago Bulls": "CHI",
    "Cleveland Cavaliers": "CLE",
    "Dallas Mavericks": "DAL",
    "Denver Nuggets": "DEN",
    "Detroit Pistons": "DET",
    "Golden State Warriors": "GSW",
    "Houston Rockets": "HOU",
    "Indiana Pacers": "IND",
    "Los Angeles Clippers": "LAC",
    "Los Angeles Lakers": "LAL",
    "Memphis Grizzlies": "MEM",
    "Miami Heat": "MIA",
    "Milwaukee Bucks": "MIL",
    "Minnesota Timberwolves": "MIN",
    "New Orleans Pelicans": "NOP",
    "New York Knicks": "NYK",
    "Oklahoma City Thunder": "OKC",
    "Orlando Magic": "ORL",
    "Philadelphia 76ers": "PHI",
    "Phoenix Suns": "PHX",
    "Portland Trail Blazers": "POR",
    "Sacramento Kings": "SAC",
    "San Antonio Spurs": "SAS",
    "Toronto Raptors": "TOR",
    "Utah Jazz": "UTA",
    "Washington Wizards": "WAS"
}

stats["Team"] = stats["Team"].map(team_map)
advanced["Team"] = advanced["Team"].map(team_map)

# Remove any rows that aren't actual NBA teams
stats = stats[stats["Team"].notna()]
advanced = advanced[advanced["Team"].notna()]

# =====================================================
# SELECT IMPORTANT COLUMNS
# =====================================================

payroll = payroll[
    [
        "Team",
        "Avg Age",
        "Total Cash",
        "Dead"
    ]
].rename(
    columns={
        "Avg Age": "AvgAge",
        "Total Cash": "Payroll",
        "Dead": "DeadCash"
    }
)

salarycap = salarycap[
    [
        "Team",
        "Team Total Cap Allocations",
        "Cap Space",
        "Dead Cap"
    ]
].rename(
    columns={
        "Team Total Cap Allocations": "CapAllocations",
        "Dead Cap": "DeadCap"
    }
)

stats = stats[
    [
        "Team",
        "PTS",
        "FG%",
        "3P%",
        "FT%",
        "TRB",
        "AST",
        "STL",
        "BLK",
        "TOV"
    ]
]

advanced = advanced[
    [
        "Team",
        "Age",
        "W",
        "L",
        "MOV",
        "SRS",
        "ORtg",
        "DRtg",
        "NRtg",
        "Pace",
        "TS%"
    ]
].rename(
    columns={
        "Age": "RosterAge"
    }
)

# =====================================================
# MERGE DATASETS
# =====================================================

df = payroll.merge(salarycap, on="Team", how="inner")

df = df.merge(stats, on="Team", how="inner")

df = df.merge(advanced, on="Team", how="inner")

# =====================================================
# FEATURE ENGINEERING
# =====================================================

df["WinPct"] = df["W"] / (df["W"] + df["L"])

df["CostPerWin"] = df["Payroll"] / df["W"]

df["PayrollRank"] = df["Payroll"].rank(
    ascending=False,
    method="min"
)

df["CapAllocationRank"] = df["CapAllocations"].rank(
    ascending=False,
    method="min"
)

df["PointDiffPerDollar"] = (
    df["MOV"] / (df["Payroll"] / 1_000_000)
)

df["NetRatingPerMillion"] = (
    df["NRtg"] / (df["Payroll"] / 1_000_000)
)

# =====================================================
# SORT BY WINS
# =====================================================

df = df.sort_values(
    by="W",
    ascending=False
)

# =====================================================
# ROUND DECIMAL COLUMNS
# =====================================================

round_cols = [
    "WinPct",
    "CostPerWin",
    "PointDiffPerDollar",
    "NetRatingPerMillion"
]

df[round_cols] = df[round_cols].round(3)

# =====================================================
# SAVE DATASET
# =====================================================

output_file = "NBA_Master_Dataset_2026.csv"
backup_file = "NBA_Master_Dataset_2025_26.csv"

df.to_csv(output_file, index=False)
df.to_csv(backup_file, index=False)

print("Dataset created successfully!")
print(df.head())

print("\nRows:", len(df))
print("Columns:", len(df.columns))

c:\Users\shank_\OneDrive\Documents\AS_Desktop\Summer2026python\summer-2026-python
Dataset created successfully!
   Team  AvgAge    Payroll  DeadCash  CapAllocations  Cap Space  DeadCap  \
19  OKC    24.6  187827634   2298085       188057857  -33410857  2296274   
29  SAS    26.6  187078708   7569181       183990669  -29343669  7258201   
14  DET    25.9  179870831   6210275       193082507  -38435507  9337154   
7   BOS    26.1  196926936    582842       194526296  -39879296   469063   
24  DEN    26.7  192006289    636435       200743895  -46096895        0   

      PTS    FG%    3P%  ...   DRtg  NRtg  Pace    TS%  WinPct   CostPerWin  \
19  119.0  0.484  0.365  ...  107.7  11.2  99.3  0.599   0.780  2934806.781   
29  119.8  0.483  0.359  ...  111.3   8.3  99.9  0.595   0.756  3017398.516   
14  117.8  0.485  0.356  ...  109.7   8.2  99.3  0.583   0.732  2997847.183   
7   114.9  0.467  0.367  ...  112.7   8.1  94.8  0.583   0.683  3516552.429   
24  122.1  0.496  0.396  ...  117.4 

In [9]:
df.to_csv(
    "NBA_Master_Dataset_2026.csv",
    index=False
)

In [10]:
import pandas as pd
df = pd.read_csv('NBA_Master_Dataset_2026.csv')

In [11]:
df.head()

,Team,AvgAge,Payroll,DeadCash,CapAllocations,Cap Space,DeadCap,PTS,FG%,3P%,...,DRtg,NRtg,Pace,TS%,WinPct,CostPerWin,PayrollRank,CapAllocationRank,PointDiffPerDollar,NetRatingPerMillion
0,OKC,24.6,187827634,2298085,188057857,-33410857,2296274,119.0,0.484,0.365,...,107.7,11.2,99.3,0.599,0.780,2934806.781,19.0,23.0,0.059,0.060
1,SAS,26.6,187078708,7569181,183990669,-29343669,7258201,119.8,0.483,0.359,...,111.3,8.3,99.9,0.595,0.756,3017398.516,21.0,25.0,0.044,0.044
2,DET,25.9,179870831,6210275,193082507,-38435507,9337154,117.8,0.485,0.356,...,109.7,8.2,99.3,0.583,0.732,2997847.183,25.0,21.0,0.045,0.046
3,BOS,26.1,196926936,582842,194526296,-39879296,469063,114.9,0.467,0.367,...,112.7,8.1,94.8,0.583,0.683,3516552.429,10.0,18.0,0.039,0.041
4,DEN,26.7,192006289,636435,200743895,-46096895,0,122.1,0.496,0.396,...,117.4,5.2,98.4,0.616,0.659,3555672.019,12.0,11.0,0.027,0.027


In [12]:
df.columns

Index(['Team', 'AvgAge', 'Payroll', 'DeadCash', 'CapAllocations', 'Cap Space',
       'DeadCap', 'PTS', 'FG%', '3P%', 'FT%', 'TRB', 'AST', 'STL', 'BLK',
       'TOV', 'RosterAge', 'W', 'L', 'MOV', 'SRS', 'ORtg', 'DRtg', 'NRtg',
       'Pace', 'TS%', 'WinPct', 'CostPerWin', 'PayrollRank',
       'CapAllocationRank', 'PointDiffPerDollar', 'NetRatingPerMillion'],
      dtype='str')

In [13]:
df.sort_values(by= "CostPerWin", ascending=False)

,Team,AvgAge,Payroll,DeadCash,CapAllocations,Cap Space,DeadCap,PTS,FG%,3P%,...,DRtg,NRtg,Pace,TS%,WinPct,CostPerWin,PayrollRank,CapAllocationRank,PointDiffPerDollar,NetRatingPerMillion
29,WAS,23.8,173401789,22865076,232033397,-77386397,23844113,112.9,0.463,0.359,...,122.7,-11.7,101.4,0.566,0.207,1.020011e+07,28.0,2.0,-0.069,-0.067
28,IND,26.0,185218474,4637120,191900207,-37253207,4440132,112.4,0.458,0.356,...,118.8,-7.9,101.0,0.568,0.232,9.748341e+06,22.0,22.0,-0.043,-0.043
26,SAC,27.3,189541488,495591,213853369,-59206369,395910,111.0,0.467,0.340,...,121.5,-10.1,99.2,0.560,0.268,8.615522e+06,16.0,5.0,-0.053,-0.053
23,DAL,26.0,220501292,6644286,200372430,-45725430,10343186,114.1,0.466,0.344,...,116.5,-5.3,101.7,0.564,0.317,8.480819e+06,2.0,12.0,-0.025,-0.024
27,BKN,23.4,144313252,23426027,150960352,3686648,24804999,105.9,0.443,0.340,...,119.0,-10.3,97.1,0.559,0.244,7.215663e+06,30.0,30.0,-0.069,-0.071
22,NOP,25.1,187462285,2118673,207453274,-52806274,5469850,115.5,0.466,0.347,...,118.9,-4.5,100.3,0.568,0.317,7.210088e+06,20.0,9.0,-0.024,-0.024
24,MEM,25.1,178201340,12942654,157650285,-3003285,24462849,114.7,0.456,0.353,...,118.8,-5.9,101.3,0.570,0.305,7.128054e+06,26.0,29.0,-0.034,-0.033
25,UTA,25.1,147637614,17400130,173117343,-18470343,25968281,117.6,0.467,0.345,...,122.3,-8.2,102.3,0.575,0.268,6.710801e+06,29.0,28.0,-0.057,-0.056
21,CHI,24.7,182641502,10844750,187673271,-33026271,16916687,116.3,0.469,0.356,...,118.1,-5.1,102.5,0.580,0.378,5.891661e+06,23.0,24.0,-0.029,-0.028
19,GSW,28.8,210019601,529164,234222725,-79575725,263940,114.6,0.461,0.356,...,115.6,-0.6,99.0,0.584,0.451,5.676205e+06,4.0,1.0,-0.003,-0.003


In [14]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

df = pd.read_csv('NBA_Master_Dataset_2026.csv')

X = df[['WinPct']]
y = df['CostPerWin']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

score = model.score(X_test, y_test)
print('Random Forest R^2 score:', round(score, 3))

predictions = model.predict(X_test.head())
print('\nActual vs predicted CostPerWin:')
for actual, predicted in zip(y_test.head().values, predictions):
    print(f'Actual: {round(actual, 0)}, Predicted: {round(predicted, 0)}')

Random Forest R^2 score: 0.73

Actual vs predicted CostPerWin:
Actual: 7215663.0, Predicted: 9059232.0
Actual: 4029728.0, Predicted: 4317533.0
Actual: 8480819.0, Predicted: 7275519.0
Actual: 4729438.0, Predicted: 4442632.0
Actual: 3813579.0, Predicted: 4204799.0


In [16]:
df.describe()

,AvgAge,Payroll,DeadCash,CapAllocations,Cap Space,DeadCap,PTS,FG%,3P%,FT%,...,DRtg,NRtg,Pace,TS%,WinPct,CostPerWin,PayrollRank,CapAllocationRank,PointDiffPerDollar,NetRatingPerMillion
count,30.000000,3.000000e+01,3.000000e+01,3.000000e+01,3.000000e+01,3.000000e+01,30.000000,30.000000,30.000000,30.000000,...,30.000000,30.000000,30.000000,30.000000,30.000000,3.000000e+01,30.000000,30.000000,30.000000,30.000000
mean,25.903333,1.900695e+08,8.676791e+06,1.979373e+08,-4.329026e+07,1.016040e+07,115.606667,0.471067,0.359333,0.782833,...,115.730000,0.050000,99.353333,0.581433,0.499967,5.251840e+06,15.500000,15.500000,-0.001533,-0.001167
std,1.238320,1.712428e+07,9.986730e+06,1.897172e+07,1.897172e+07,1.137339e+07,3.298687,0.013193,0.013368,0.021948,...,3.502034,6.241615,2.113786,0.014231,0.168850,2.016508e+06,8.803408,8.803408,0.034756,0.034934
min,23.400000,1.443133e+08,1.103000e+05,1.509604e+08,-7.957572e+07,0.000000e+00,105.900000,0.443000,0.340000,0.730000,...,107.700000,-11.700000,94.800000,0.559000,0.207000,2.934807e+06,1.000000,1.000000,-0.069000,-0.071000
25%,25.100000,1.832857e+08,6.794128e+05,1.890184e+08,-5.306406e+07,4.031005e+05,114.225000,0.462250,0.350000,0.766750,...,113.550000,-5.250000,97.775000,0.570000,0.332250,3.994259e+06,8.250000,8.250000,-0.028000,-0.027000
50%,25.950000,1.897087e+08,3.467602e+06,1.979399e+08,-4.329294e+07,4.954991e+06,115.600000,0.467500,0.359000,0.780500,...,115.300000,1.350000,99.350000,0.580500,0.543000,4.375618e+06,15.500000,15.500000,0.007000,0.007000
75%,26.700000,1.985538e+08,1.628576e+07,2.077111e+08,-3.437144e+07,2.430816e+07,117.750000,0.481750,0.366500,0.799000,...,118.625000,4.775000,100.875000,0.589750,0.634000,6.506016e+06,22.750000,22.750000,0.024000,0.024750
max,28.800000,2.274005e+08,2.850821e+07,2.342227e+08,3.686648e+06,2.948342e+07,122.100000,0.502000,0.396000,0.823000,...,122.700000,11.200000,103.400000,0.616000,0.780000,1.020011e+07,30.000000,30.000000,0.059000,0.060000
